**Purpose:** `v1_bert_embeddings.ipynb` — part of `02_sentiment/news` (see the stage README).

**Inputs:** `/kaggle/input/news-sentiment/dataset.parquet`

**Outputs:** `v11_{hyp_i}.csv`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.seeds import SEED_SENTIMENT_NEWS_V1_BERT_EMBEDDINGS


In [ ]:
!pip install torch==2.2.2+cu118 torchvision==0.17.2+cu118 torchaudio==2.2.2+cu118 --index-url https://download.pytorch.org/whl/cu118

!pip install transformers==4.36.2 scikit-learn pandas numpy


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected")


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import json
import random
import time

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    precision_recall_fscore_support
)
from sklearn.metrics import f1_score
from collections import Counter
from sklearn.model_selection import StratifiedKFold


# =========================================================
# 3. Load & Balance Dataset : train set only
# =========================================================
df_raw = pd.read_parquet("/kaggle/input/news-sentiment/dataset.parquet")

with open("/kaggle/input/news-sentiment/sector_sentiment_quotations.json", "r") as f:
    sector_sentiment_quotations = json.load(f)

gpt_testset_ids = []
for sector in sector_sentiment_quotations:
    for expected_sentiment in sector_sentiment_quotations[sector]:
        for _ in sector_sentiment_quotations[sector][expected_sentiment]:
            for quote in sector_sentiment_quotations[sector][expected_sentiment][_]:
                if quote["label"] == expected_sentiment:
                    gpt_testset_ids.append(quote["gkrecordid"] + " (" + sector + ")")


train_df = df_raw[~df_raw["GKGRECORDID"].isin(gpt_testset_ids)].copy()
min_count = train_df.value_counts(["sector", "majority_sentiment"]).values.min()
df = (
    train_df
    .groupby(["sector", "majority_sentiment"])
    .apply(lambda x: x.sample(min_count, random_state=SEED_SENTIMENT_NEWS_V1_BERT_EMBEDDINGS))
    .reset_index(drop=True)
)

# =========================================================
# 1. Configuration
# =========================================================
SEARCH_SPACE = [ # batch_size, max_len, epochs, dropout, num_heads, lr_head, weight_decay_head, lr_backbone, weight_decay_backbone

    # ---- Baseline / safest ----
    [16, 256, 5, 0.3, 12, 3e-4, 1e-3, 1e-5, 1e-2],

    # ---- Slightly longer training ----
    [16, 256, 6, 0.3, 12, 3e-4, 1e-3, 1e-5, 1e-2],

    # ---- More regularization (noisy folds) ----
    [16, 256, 6, 0.4, 12, 2e-4, 2e-3, 8e-6, 1e-2],

    # ---- Higher head LR (hard labels) ----
    [16, 256, 5, 0.3, 12, 5e-4, 1e-3, 1e-5, 1e-2],

    # ---- Larger batch (if GPU allows) ----
    [32, 256, 5, 0.3, 12, 3e-4, 1e-3, 1e-5, 1e-2],

    # ---- Conservative backbone (high CV variance) ----
    [16, 256, 7, 0.3, 12, 2e-4, 1e-3, 6e-6, 1e-2],
]

all_results_dict = {}

start_time = time.time()

In [ ]:

for hyp_i in range(len(SEARCH_SPACE)):

    CONFIG_list = SEARCH_SPACE[hyp_i]
    CONFIG = {
        "batch_size": int(CONFIG_list[0]),
        "max_len": int(CONFIG_list[1]),
        "epochs": int(CONFIG_list[2]),
        "dropout": float(CONFIG_list[3]),
        "num_heads": int(CONFIG_list[4]),
        "lr_head": float(CONFIG_list[5]),
        "weight_decay_head": float(CONFIG_list[6]),
        "lr_backbone": float(CONFIG_list[7]),
        "weight_decay_backbone": float(CONFIG_list[8]),
    }

    print(f"\n=== Hyperparameter Set {hyp_i}/5 ===")
    print(CONFIG)


    # =========================================================
    # 4. Encode Labels & Sectors
    # =========================================================
    label_encoder = LabelEncoder()
    df["label"] = label_encoder.fit_transform(df["majority_sentiment"])

    sector_encoder = LabelEncoder()
    df["sector_id"] = sector_encoder.fit_transform(df["sector"])

    NUM_LABELS = len(label_encoder.classes_)
    NUM_SECTORS = len(sector_encoder.classes_)

    # =========================================================
    # 5. Sector Descriptions
    # =========================================================

    # https://www.msci.com/downloads/documents/indexes/gics/GICS+Sector+Definitions+2023.pdf

    SECTOR_DESCRIPTIONS = {
        "Energy": (
            "The Energy Sector comprises companies engaged in exploration & "
            "production, refining & marketing, and storage & transportation of oil "
            "and gas and coal & consumable fuels. It also includes companies that "
            "offer oil & gas equipment and services."
        ),
        "Materials": (
            "The Materials Sector includes companies that manufacture chemicals, "
            "construction materials, forest products, glass, paper and related "
            "packaging products, and metals, minerals and mining companies, "
            "including producers of steel."
        ),
        "Industrials": (
            "The Industrials Sector includes manufacturers and distributors of "
            "capital goods such as aerospace & defense, building products, "
            "electrical equipment and machinery and companies that offer "
            "construction & engineering services. It also includes providers of "
            "commercial & professional services and transportation services."
        ),
        "Consumer Discretionary": (
            "The Consumer Discretionary Sector encompasses those businesses that "
            "tend to be the most sensitive to economic cycles. Its manufacturing "
            "segment includes automobiles & components, household durable goods, "
            "leisure products, and textiles & apparel. The services segment "
            "includes hotels, restaurants, and other leisure facilities, as well as "
            "distributors and retailers of consumer discretionary products."
        ),
        "Consumer Staples": (
            "The Consumer Staples Sector comprises companies whose businesses are "
            "less sensitive to economic cycles. It includes manufacturers and "
            "distributors of food, beverages and tobacco and producers of non-durable "
            "household goods and personal products. It also includes distributors "
            "and retailers of consumer staples products including food & drug "
            "retailing companies."
        ),
        "Health Care": (
            "The Health Care Sector includes health care providers and services, "
            "companies that manufacture and distribute health care equipment and "
            "supplies, health care technology firms, and companies involved in "
            "pharmaceuticals and biotechnology research, development, production, "
            "and marketing."
        ),
        "Financials": (
            "The Financials Sector contains companies engaged in banking, financial "
            "services, consumer finance, capital markets, and insurance activities. "
            "It also includes financial exchanges, data services, and mortgage real "
            "estate investment trusts."
        ),
        "Information Technology": (
            "The Information Technology Sector comprises companies offering "
            "software and technology services, and manufacturers and distributors "
            "of technology hardware and equipment including communications "
            "equipment, computers, peripherals, and semiconductor products."
        ),
        "Communication Services": (
            "The Communication Services Sector includes companies that facilitate "
            "communication and offer related content and information through various "
            "mediums, including telecommunications, media and entertainment, and "
            "interactive services and platforms."
        ),
        "Utilities": (
            "The Utilities Sector comprises utility companies such as electric, "
            "gas, and water utilities, as well as independent power producers, "
            "energy traders, and companies involved in renewable generation."
        ),
        "Real Estate": (
            "The Real Estate Sector contains companies engaged in real estate "
            "development and operations as well as firms offering real estate-"
            "related services, including equity real estate investment trusts."
        ),
    }

    sector_texts = [SECTOR_DESCRIPTIONS[s] for s in sector_encoder.classes_]

    # =========================================================
    # 6. Tokenizer
    # =========================================================
    tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

    sector_encodings = tokenizer(
        sector_texts,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    # =========================================================
    # 7. Dataset
    # =========================================================
    class SentimentDataset(Dataset):
        def __init__(self, df):
            self.df = df.reset_index(drop=True)

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]

            enc = tokenizer(
                row["quotation"],
                padding="max_length",
                truncation=True,
                max_length=CONFIG["max_len"],
                return_tensors="pt"
            )

            return {
                "input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "sector_id": torch.tensor(int(row["sector_id"]), dtype=torch.long),
                "label": torch.tensor(int(row["label"]), dtype=torch.long),
            }

    # =========================================================
    # 8. Model
    # =========================================================
    class FinBERTSectorTextAttention(nn.Module):
        def __init__(self, num_labels, sector_encodings):
            super().__init__()

            self.finbert = AutoModel.from_pretrained("ProsusAI/finbert")
            hidden = self.finbert.config.hidden_size

            self.register_buffer("sector_input_ids", sector_encodings["input_ids"])
            self.register_buffer("sector_attention_mask", sector_encodings["attention_mask"])

            self.attention = nn.MultiheadAttention(
                embed_dim=hidden,
                num_heads=CONFIG["num_heads"],
                batch_first=True
            )

            self.dropout = nn.Dropout(CONFIG["dropout"])
            self.classifier = nn.Linear(hidden, num_labels)

        def forward(self, input_ids, attention_mask, sector_id):
            text_out = self.finbert(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            token_embs = text_out.last_hidden_state

            sector_ids = self.sector_input_ids[sector_id]
            sector_mask = self.sector_attention_mask[sector_id]

            sector_out = self.finbert(
                input_ids=sector_ids,
                attention_mask=sector_mask
            )

            sector_emb = sector_out.last_hidden_state[:, 0].unsqueeze(1)

            attended, _ = self.attention(
                query=sector_emb,
                key=token_embs,
                value=token_embs,
                key_padding_mask=~attention_mask.bool()
            )

            pooled = self.dropout(attended.squeeze(1))
            return self.classifier(pooled)



    all_fold_dfs = []



    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED_SENTIMENT_NEWS_V1_BERT_EMBEDDINGS
    )

    for fold, (tr_idx, val_idx) in enumerate(
        skf.split(df, df["label"]), 1
    ):
        print(f"\n===== Fold {fold}/5 =====")

        tr_df = df.iloc[tr_idx]
        val_df = df.iloc[val_idx]

        train_loader = DataLoader(
            SentimentDataset(tr_df),
            batch_size=CONFIG["batch_size"],
            shuffle=True
        )

        val_loader = DataLoader(
            SentimentDataset(val_df),
            batch_size=CONFIG["batch_size"]
        )


        # =========================================================
        # 10. Training Setup
        # =========================================================
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        model = FinBERTSectorTextAttention(NUM_LABELS, sector_encodings).to(device)

        for p in model.finbert.parameters():
            p.requires_grad = False

        for name, p in model.finbert.named_parameters():
            if any(f"encoder.layer.{i}." in name for i in [10, 11]):
                p.requires_grad = True

        # optimizer = torch.optim.AdamW(
        #     filter(lambda p: p.requires_grad, model.parameters()),
        #     lr=CONFIG["lr"]
        # )

        head_params = list(model.attention.parameters()) + list(model.classifier.parameters())

        last2_params = [
            p for p in model.finbert.parameters() if p.requires_grad
        ]

        assert len(head_params) > 0, "No head parameters found!"
        assert len(last2_params) > 0, "No backbone parameters found!"


        optimizer = torch.optim.AdamW(
            [
                {"params": head_params, "lr": CONFIG["lr_head"], "weight_decay": CONFIG["weight_decay_head"]},
                {"params": last2_params, "lr": CONFIG["lr_backbone"], "weight_decay": CONFIG["weight_decay_backbone"]},
            ]
        )

        criterion = nn.CrossEntropyLoss()

        # =========================================================
        # 11. Evaluation Utilities
        # =========================================================
        def evaluate(model, loader):
            model.eval()
            preds, labels, sectors = [], [], []

            with torch.no_grad():
                for batch in loader:
                    logits = model(
                        batch["input_ids"].to(device),
                        batch["attention_mask"].to(device),
                        batch["sector_id"].to(device)
                    )
                    preds.extend(torch.argmax(logits, 1).cpu().tolist())
                    labels.extend(batch["label"].cpu().tolist())
                    sectors.extend(batch["sector_id"].cpu().tolist())


            return np.array(preds), np.array(labels), np.array(sectors)

        def macro_f1_and_support(labels, preds, num_classes):
            macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)

            counts = Counter(labels)
            support = [counts.get(i, 0) for i in range(num_classes)]

            return macro_f1, support

        def build_fold_sector_df(
            fold_id,
            train_preds, train_labels, train_sectors,
            val_preds, val_labels, val_sectors
        ):
            rows = []

            for sid in np.unique(val_sectors):
                sector_name = sector_encoder.inverse_transform([sid])[0]

                # --- Train ---
                train_mask = train_sectors == sid
                mf1_train, _ = macro_f1_and_support(
                    train_labels[train_mask],
                    train_preds[train_mask],
                    NUM_LABELS
                )

                # --- Validation ---
                val_mask = val_sectors == sid
                mf1_val, sup_val = macro_f1_and_support(
                    val_labels[val_mask],
                    val_preds[val_mask],
                    NUM_LABELS
                )

                rows.append({
                    "sector": sector_name,
                    f"MacroF1_train_{fold_id}": mf1_train,
                    f"MacroF1_val_{fold_id}": mf1_val,
                    f"Sup_val_{fold_id}": sup_val,
                })

            # -------- GLOBAL --------
            mf1_train_g, _ = macro_f1_and_support(
                train_labels, train_preds, NUM_LABELS
            )
            mf1_val_g, sup_val_g = macro_f1_and_support(
                val_labels, val_preds, NUM_LABELS
            )

            rows.append({
                "sector": "GLOBAL",
                f"MacroF1_train_{fold_id}": mf1_train_g,
                f"MacroF1_val_{fold_id}": mf1_val_g,
                f"Sup_val_{fold_id}": sup_val_g,
            })

            return pd.DataFrame(rows)

        # =========================================================
        # 12. Training Loop
        # =========================================================
        for epoch in range(CONFIG["epochs"]):

            runned_time = time.time() - start_time
            print(f"Time elapsed: {runned_time/60:.2f} minutes")

            if runned_time > 11 * 60 * 60:
                print("Stopping training due to time limit.")
                raise

            model.train()
            epoch_loss = 0.0

            for batch in train_loader:
                optimizer.zero_grad()

                logits = model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["sector_id"].to(device)
                )

                loss = criterion(logits, batch["label"].to(device))
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()

            avg_loss = epoch_loss / len(train_loader)
            print(f"Epoch {epoch+1}/{CONFIG['epochs']} | Loss: {avg_loss:.4f}")

        # =========================================================
        # 13. Final Evaluation & Logging
        # =========================================================
        train_preds, train_labels, train_sectors = evaluate(model, train_loader)
        val_preds, val_labels, val_sectors = evaluate(model, val_loader)

        fold_df = build_fold_sector_df(
            fold_id=fold + 1,
            train_preds=train_preds,
            train_labels=train_labels,
            train_sectors=train_sectors,
            val_preds=val_preds,
            val_labels=val_labels,
            val_sectors=val_sectors,
        )

        all_fold_dfs.append(fold_df)

    final_df = all_fold_dfs[0]
    for df_next in all_fold_dfs[1:]:
        final_df = final_df.merge(df_next, on="sector", how="left")


    final_df.to_csv(f"v11_{hyp_i}.csv", index=False)
    data_dict = json.loads(final_df.to_json(orient="split"))

    all_results_dict[" | ".join(map(str, CONFIG_list))] = data_dict
    with open(f"results_{time.time()}.json", "w") as f:
        json.dump(all_results_dict, f)